# Sample Training Data

Here, I sample training data to make each sample more indepdent. Spatial data contain many redundancies !

Follow YOG exmaple!

In [7]:
from global_vars import *
import numpy as np
import pandas as pd
import gc
import xarray as xr
import os
import glob
from datetime import datetime, timedelta
from tqdm import tqdm
from pathlib import Path

src_folder = os.path.join(data_folder, 'nodeep_1deg_clean/')
output_folder = os.path.join(data_folder, 'nodeep_train_data/')
output_folder_test = os.path.join(data_folder, 'nodeep_test_data/')

start1 = startdate_dyamond1
start2 = startdate_dyamond2

end1 = startdate_dyamond1 + timedelta(days=35)
end2 = startdate_dyamond2 + timedelta(days=35)


def is_between(date_str, start, end, fmt="%Y-%m-%d"):
    d     = datetime.strptime(date_str,  fmt)
    return start <= d <= end


def isin_training_data(date_str):
    return is_between(date_str, start1, end1) or is_between(date_str, start2, end2)

### SAVE ALL DATA FROM THE LAST 4 DAYS OF EACH SIMULATION FOR TESTING !

### DO NOT worry about the test data for now !!!! I will deal with it later !!!!


src_files = glob.glob(os.path.join(src_folder, '*.nc'))
print(len(src_files))

501


In [12]:
for file in tqdm(src_files):
    filename = os.path.basename(file)
    output_file = Path(os.path.join(output_folder, filename)).with_suffix(".csv")

    ds = xr.open_dataset(file)
    stacked = ds.stack(col=("lat", "lon"))
    
    new_vars = {}
    for name, da in stacked.data_vars.items():
        if "lev" in da.dims:
            for i, lev_val in enumerate(stacked.lev.values):
                slab = da.isel(lev=i).drop_vars("lev")     # -> (col,)
                slab.attrs = {**da.attrs, "lev": float(lev_val)}
                new_vars[f"{name}_lev{i}"] = slab
        elif "ilev" in da.dims:
            for i, lev_val in enumerate(stacked.ilev.values):
                slab = da.isel(ilev=i).drop_vars("ilev")     # -> (col,)
                slab.attrs = {**da.attrs, "ilev": float(lev_val)}
                new_vars[f"{name}_ilev{i}"] = slab
        else:
            new_vars[name] = da                            # already (col,)
    
    ds_flat = xr.Dataset(new_vars)
    
    df = ds_flat.to_dataframe()

    if not isin_training_data(filename[29:39]):
        ### If file is not in training set, don't sample and instead just move it directly to test data folder
        output_file = Path(os.path.join(output_folder_test, filename)).with_suffix(".csv")
        df.to_csv(output_file)
        continue
    
    
    ### SAMPLE 20 LONS FOR EACH LAT
    sampled = df.groupby(level="lat", group_keys=False).sample(n=20, random_state=0)
    sampled.to_csv(output_file)

100%|██████████| 501/501 [52:43<00:00,  6.31s/it]  
